# Fencing bout segmenter

Colab driver for the `fenceseg` package. All logic lives in the package so it
can be version-controlled and tested outside a notebook; this notebook just
wires it up for Colab.

**The start and end conditions are unchanged from the original notebook.**
`tools/verify_state_machine.py` fuzzes the refactored state machine against a
verbatim transcription of the original over ~26,000 sequences and requires an
exact match.

**Before running anything:** `Runtime -> Change runtime type -> T4 GPU` (or
better). Everything below assumes a GPU runtime.

## 1. Check the runtime, mount Drive

Colab's local disk (`/content`) is wiped when the runtime disconnects or
recycles. Mount Drive so downloaded streams, cut bouts and the analysis
report survive a session — you do not want to re-download a 3-hour stream
because the runtime recycled.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/fencing')  # adjust if you like
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Get the `fenceseg` code onto the machine

Colab starts from a bare VM each session — there is no dataset mount like
Kaggle's `/kaggle/input`. Two options; pick one.

In [ ]:
# Option A: upload the project zip (fencing-segmenter.zip) via the file picker.
from google.colab import files
uploaded = files.upload()   # select fencing-segmenter.zip in the dialog

import zipfile
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('/content/fencing-segmenter')
print('extracted to /content/fencing-segmenter')

In [ ]:
# Option B: clone from GitHub instead, once the repo actually contains the
# fenceseg/ folder (not flattened files at the root). Skip Option A above if
# you use this.
# !git clone https://github.com/iradosla0/feningocr.git /content/fencing-segmenter

## 3. Install dependencies

Colab already ships `torch` pre-linked to its CUDA build — do **not**
`pip install torch`, a version-unconstrained reinstall can silently swap in a
CPU-only or mismatched-CUDA wheel and break GPU inference. `opencv`,
`pandas`, `pillow` and `ffmpeg` are preinstalled too. Only the genuinely
missing pieces get installed here.

In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr
!pip -q install yt-dlp pytesseract seaborn

In [ ]:
import sys
REPO = pathlib.Path('/content/fencing-segmenter')
sys.path.insert(0, str(REPO))

from fenceseg.config import Config
from fenceseg.pipeline import analyse, build_bouts, write_report, process
from fenceseg.cut import cut_all

## 4. Verify the conditions are untouched

Should print `PASS`. Re-run this any time you edit `fenceseg/segment.py`.

In [ ]:
!cd /content/fencing-segmenter && python tools/verify_state_machine.py

## 5. Get the weights and the video

Small files (`best.pt`) are easiest via the upload dialog. For the stream
itself, either download straight from FencingTV or point at a file already on
Drive.

**FencingTV URL**: F12 -> Network tab -> filter `.m3u8` -> click the entry
with the random-looking name (**not** the ones labelled `rendition`) ->
Headers tab -> copy the Request URL.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select best.pt
weights_path = pathlib.Path('/content') / next(iter(uploaded))
print(weights_path)

# Or, if best.pt already lives on Drive:
# weights_path = DRIVE_ROOT / 'best.pt'

In [ ]:
from fenceseg.download import download

URL = "PASTE_THE_M3U8_REQUEST_URL_HERE"
video = download(URL, DRIVE_ROOT / 'work' / 'stream_01.mp4',
                 concurrent_fragments=8)
print(video)

In [ ]:
# Already have a local file (on Drive or freshly uploaded)? Skip the cell
# above and point at it directly instead.
# video = DRIVE_ROOT / 'stream_01.mp4'

## 6. Configure

`hwaccel` is set to `None` below. Colab's preinstalled `ffmpeg` build does not
reliably include CUDA-accelerated decode, unlike the software decode path
which always works. Check what your runtime actually has before turning it
on:

In [ ]:
!ffmpeg -hwaccels

In [ ]:
cfg = Config(
    weights = weights_path,
    workdir = DRIVE_ROOT / 'work',
    outdir  = DRIVE_ROOT / 'bouts',

    sample_fps = 1.0,        # decision rate; 1.0 == the original notebook
    batch_size = 32,         # throughput only, does not affect decisions
    hwaccel    = None,       # set to 'cuda' only if it appeared in -hwaccels above
    device     = 'cuda:0',
    half       = True,

    use_templates = True,    # learned digit templates: the big OCR speedup
    temporal_vote = False,   # see README before enabling

    cut_mode = 'copy',       # 'reencode' for frame-accurate cuts
)
cfg

## 7. Analyse, then cut

One pass over the video collects scores, fencer counts and name plates;
boundaries and filenames are resolved afterwards.

In [ ]:
records, boundaries, stats = analyse(video, cfg)
bouts = build_bouts(records, boundaries, cfg)
write_report(video, records, boundaries, bouts, stats, cfg)

print(f"\n{len(bouts)} bouts\n")
for b in bouts:
    print(f"{b.start:8.1f} -> {b.end:8.1f}   {b.filename}")

### Check the names before cutting

If a filename looks wrong, the raw plate text is on the bout object. Fixing a
name here is much cheaper than re-cutting.

In [ ]:
for b in bouts:
    print(f"{b.index:3d}  L={b.left_plate!r}  R={b.right_plate!r}")

# Manual override example:
# bouts[3].filename = "Alexander Massialas USA vs. Race Imboden USA"

In [ ]:
jobs = [(b.start, b.end, b.filename) for b in bouts]
written = cut_all(video, jobs, cfg.outdir, cfg.cut_mode)
print(f"\n{len(written)} files written to {cfg.outdir}")

Bouts are written straight to Drive (`cfg.outdir`), so they survive the
session ending. If you'd rather download the batch directly instead of
digging through Drive:

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/bouts', 'zip', cfg.outdir)
files.download(archive)

## 8. Optional: harvest frames to improve the detector

Writes the frames the model is least sure about, with pre-filled labels.
See `training/README.md`.

In [ ]:
!cd /content/fencing-segmenter && python tools/harvest_frames.py \
    {video} --weights {cfg.weights} --out /content/drive/MyDrive/fencing/dataset/candidates \
    --per-bucket 150